<a href="https://colab.research.google.com/github/ailoveu14333-ops/AI-Machine-Proj/blob/main/NBCP_Qwen_Final_na.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Setup and Install Dependencies


In [1]:
!pip install -q torch transformers accelerate
!pip install -q --upgrade langchain faiss-cpu pypdf sentence-transformers langchain-community
!pip install -q rouge-score pandas

import re
import torch
import numpy as np
import pandas as pd
from collections import Counter

from transformers import AutoTokenizer, AutoModelForCausalLM
from pypdf import PdfReader

from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import faiss

from rouge_score import rouge_scorer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 37.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.4/331.4 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
  Preparing metadata (setup.py) ... done


# Load Qwen

In [2]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

model.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((1536,), eps=1e-06)
    (rotar

# Loading NBCP

In [3]:
from pypdf import PdfReader
import re

pdf_path = None
try:
    from google.colab import files
    uploaded = files.upload()
    pdf_path = list(uploaded.keys())[0]
except Exception as e:
    print("Upload skipped or not in Colab:", e)

if pdf_path is None:
    raise ValueError("pdf_path not set. Upload a PDF or set pdf_path manually.")

reader = PdfReader(pdf_path)

pages = []
for i, page in enumerate(reader.pages, start=1):
    text = page.extract_text() or ""
    pages.append({"page": i, "text": text})

print("Pages loaded:", len(pages))
print("First page preview:", (pages[0]["text"][:300] if pages and pages[0]["text"] else "EMPTY"))

Saving NBCP.pdf to NBCP.pdf
Pages loaded: 61
First page preview: The National Building Code (P.D. 1096) 
1 
 
NATIONAL BUILDING CODE  
OF THE PHILIPPINES  
 
 
MALACAÑANG 
Manila 
 
 
PRESIDENTIAL DECREE NO. 1096 
 
ADOPTING A NATIONAL BUILDING CODE OF THE PHILIPPINES THEREBY 
REVISING REPUBLIC ACT NUMBERED SIXTY-FIVE HUNDRED FORTY ONE 
 
 
 
 WHEREAS, the countr


# Text Cleaning + Chunking

In [4]:
def clean_text_minimal(text: str) -> str:
    if not text:
        return ""
    # Keep content; just normalize whitespace
    text = text.replace("\r", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text

# Clean each page
for p in pages:
    p["clean_text"] = clean_text_minimal(p["text"])

# Split into chunks while keeping page metadata
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=450,
    chunk_overlap=200
)

chunks = []
chunk_meta = []  # aligns with chunks list

for p in pages:
    if not p["clean_text"]:
        continue
    page_chunks = text_splitter.split_text(p["clean_text"])
    for c in page_chunks:
        chunks.append(c)
        chunk_meta.append({"page": p["page"]})

print("Total chunks:", len(chunks))
print("Sample chunk:", chunks[0][:300] if chunks else "NO CHUNKS")

Total chunks: 271
Sample chunk: The National Building Code (P.D. 1096) 1 NATIONAL BUILDING CODE OF THE PHILIPPINES MALACAÑANG Manila PRESIDENTIAL DECREE NO. 1096 ADOPTING A NATIONAL BUILDING CODE OF THE PHILIPPINES THEREBY REVISING REPUBLIC ACT NUMBERED SIXTY-FIVE HUNDRED FORTY ONE WHEREAS, the country’s accelerating economic and 


# Embeddings + FAISS Index

In [5]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = embedding_model.encode(
    chunks,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,   # helps cosine-like retrieval
)

dimension = embeddings.shape[1]
faiss_index = faiss.IndexFlatIP(dimension)  # inner product works well w/ normalized embeddings
faiss_index.add(embeddings)

print("Embeddings shape:", embeddings.shape)
print("FAISS index size:", faiss_index.ntotal)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/9 [00:00<?, ?it/s]

Embeddings shape: (271, 384)
FAISS index size: 271


# RAG

In [22]:
def retrieve_relevant_chunks(query: str, top_k: int = 5):
    q_emb = embedding_model.encode([query], convert_to_numpy=True, normalize_embeddings=True)
    scores, idxs = faiss_index.search(q_emb, top_k)

    results = []
    for score, idx in zip(scores[0], idxs[0]):
        if idx < 0:
            continue
        results.append({
            "text": chunks[idx],
            "page": chunk_meta[idx]["page"],
            "score": float(score),
            "chunk_index": int(idx),
        })
    return results

# Prompt Engineering

In [23]:
def build_prompt(context_items, question: str) -> str:
    context_str = "\n\n".join([f"[Page {c['page']}] {c['text']}" for c in context_items])

    return f"""
You are answering using ONLY the context.

Rules:
- Output ONE line only.
- If the exact answer is not in context, output exactly:
Not specified in the document.

- Do not add notes, explanations, or extra commentary.
- Do not paraphrase numbers; keep exact numbers and units.
- If answering with a value, copy the exact clause/sentence from context that contains the value.

Context:
{context_str}

Question: {question}
Answer:
""".strip()

# Generating Answer

In [24]:
def keyword_gate(question: str, contexts: list[dict], keep: int = 4):
    q = question.lower()
    # extract rough keywords (keep words length>=4 + numbers)
    kws = set(re.findall(r"[a-z]{4,}|\d+(?:\.\d+)?", q))
    scored = []
    for c in contexts:
        t = c["text"].lower()
        hit = sum(1 for k in kws if k in t)
        scored.append((hit, c))
    scored.sort(key=lambda x: (x[0], x[1]["score"]), reverse=True)
    return [c for hit, c in scored if hit > 0][:keep]

In [25]:
def ask_chatbot(question: str, top_k: int = 20, max_new_tokens: int = 120):
    context_items = retrieve_relevant_chunks(question, top_k=top_k)
    context_items = keyword_gate(question, context_items, keep=4)
    prompt = build_prompt(context_items, question)

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True
    )
    if torch.cuda.is_available():
        inputs = {k: v.to(model.device) for k, v in inputs.items()}

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,  # deterministic for eval
        eos_token_id=tokenizer.eos_token_id
    )

    full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Extract final answer segment
    if "FINAL ANSWER:" in full_output:
        answer = full_output.split("FINAL ANSWER:")[-1].strip()
    else:
        answer = full_output.strip()

    return {
        "answer": answer,
        "contexts": context_items,
        "prompt": prompt,  # optional: useful for debugging
        "full_output": full_output,
    }

# Chat

In [28]:
while True:
    user_input = input("Ask NBCP Bot (type 'exit' to stop): ")
    if user_input.lower() == "exit":
        print("Chatbot stopped.")
        break

    out = ask_chatbot(user_input, top_k=5)
    print("\nAnswer:\n", out["answer"], "\n")

Ask NBCP Bot (type 'exit' to stop): what is the minimum ceiling height?

Answer:
 You are answering using ONLY the context.

Rules:
- Output ONE line only.
- If the exact answer is not in context, output exactly:
Not specified in the document.

- Do not add notes, explanations, or extra commentary.
- Do not paraphrase numbers; keep exact numbers and units.
- If answering with a value, copy the exact clause/sentence from context that contains the value.

Context:
[Page 20] minimum horizontal dimension of court shall be not less than 2.00 meters. (b) All inner courts shall be connected to a street or yard, either by a passageway with a minimum width of 1.20 meters or by a door through a room or rooms. SECTION 805. Ceiling Heights (a) Habitable rooms provided with artificial ventilation shall have ceiling heights not less than 2.40 meters measured from the floor to the ceiling; Provided that for buildings of more than one - storey, the minimum ceiling height of the first storey shall be 2

# Evaluation

In [29]:
CRITICAL_WORDS = {
    "not", "no", "nor", "never",
    "shall", "must", "may", "should",
    "minimum", "maximum",
    "except", "unless", "provided", "where"
}

def _normalize_unicode(s: str) -> str:
    return (
        s.replace("²", "2")
         .replace("³", "3")
         .replace("°", " deg ")
         .replace("º", " deg ")
    )

def tokenize_code_answer(s: str) -> list[str]:
    if s is None:
        return []
    s = _normalize_unicode(str(s).lower().strip())

    token_pattern = r"""
        \d+(?:\.\d+)?%?                 # numbers like 3, 3.5, 5%
        |[a-z]+(?:-[a-z]+)*             # words / hyphenated words
        |[a-z]{1,6}/[a-z0-9]{1,6}       # unit ratios like kn/m2
    """
    return re.findall(token_pattern, s, flags=re.VERBOSE)

def token_prf_f1(pred: str, gold: str):
    pred_toks = tokenize_code_answer(pred)
    gold_toks = tokenize_code_answer(gold)

    if not pred_toks and not gold_toks:
        return 1.0, 1.0, 1.0
    if not pred_toks or not gold_toks:
        return 0.0, 0.0, 0.0

    pred_c = Counter(pred_toks)
    gold_c = Counter(gold_toks)
    common = pred_c & gold_c
    num_same = sum(common.values())

    precision = num_same / len(pred_toks)
    recall = num_same / len(gold_toks)
    f1 = 0.0 if (precision + recall) == 0 else 2 * precision * recall / (precision + recall)
    return precision, recall, f1

def normalize_for_em(s: str) -> str:
    if s is None:
        return ""
    s = _normalize_unicode(str(s).lower().strip())
    s = re.sub(r"\s+", " ", s)
    return s

def exact_match(pred: str, gold: str) -> float:
    return float(normalize_for_em(pred) == normalize_for_em(gold))

_NUM_RE = re.compile(r"\d+(?:\.\d+)?")
def extract_numbers(s: str) -> list[str]:
    if s is None:
        return []
    s = _normalize_unicode(str(s)).replace(",", "")
    return _NUM_RE.findall(s)

def numeric_recall(pred: str, gold: str) -> float:
    p = set(extract_numbers(pred))
    g = set(extract_numbers(gold))
    if not g and not p:
        return 1.0
    if not g:
        return 1.0
    if not p:
        return 0.0
    return len(p & g) / len(g)

UNIT_PATTERNS = [
    r"\bmm\b", r"\bcm\b", r"\bm\b", r"\bkm\b",
    r"\bm2\b", r"\bm3\b",
    r"\bkn\b", r"\bn\b",
    r"\bkpa\b", r"\bmpa\b", r"\bpa\b",
    r"\bpsi\b",
    r"\bdeg\b",
    r"%",
]
UNIT_RE = re.compile("|".join(UNIT_PATTERNS), flags=re.IGNORECASE)

def extract_units(s: str) -> set[str]:
    if s is None:
        return set()
    s = _normalize_unicode(str(s).lower())
    return {m.lower() for m in UNIT_RE.findall(s)}

def unit_recall(pred: str, gold: str) -> float:
    p = extract_units(pred)
    g = extract_units(gold)
    if not g and not p:
        return 1.0
    if not g:
        return 1.0
    if not p:
        return 0.0
    return len(p & g) / len(g)

rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=False)
def rouge_l_f(pred: str, gold: str) -> float:
    scores = rouge.score(gold or "", pred or "")
    return float(scores["rougeL"].fmeasure)

def post_process(answer: str) -> str:
    # Keep this minimal; DO NOT truncate on "." (it can destroy values/units!)
    if answer is None:
        return ""
    return re.sub(r"\s+", " ", answer.strip())


# ---------- Test Data ----------
test_data = [
    {
        "question": "What is the minimum ceiling height for habitable rooms?",
        "answer": "minimum ceiling height 2.70 meters"
    },
    {
        "question": "What is the required setback for buildings along public roads?",
        "answer": "The required setback for buildings along public roads is 3."
    },
    {
        "question": "Who enforces the National Building Code of the Philippines?",
        "answer": "Not specified in the document"
    },
]


# ---------- Run Evaluation ----------
rows = []
for i, item in enumerate(test_data):
    q = item["question"]
    gold = item["answer"]

    out = ask_chatbot(q, top_k=5)
    raw_pred = out["answer"]
    pred = post_process(raw_pred)

    prec, rec, f1 = token_prf_f1(pred, gold)
    row = {
        "idx": i,
        "question": q,
        "ground_truth": gold,
        "prediction": pred,
        "precision": prec,
        "recall": rec,
        "f1": f1,
        "rougeL_f": rouge_l_f(pred, gold),
        "exact_match": exact_match(pred, gold),
        "numeric_recall": numeric_recall(pred, gold),
        "unit_recall": unit_recall(pred, gold),
        "retrieved_pages": [c["page"] for c in out["contexts"]],
    }
    rows.append(row)

df_results = pd.DataFrame(rows)

print("=== FINAL EVALUATION RESULTS (Revised) ===")
print("Count:", len(df_results))
print("Avg Precision:", round(df_results["precision"].mean(), 3))
print("Avg Recall:", round(df_results["recall"].mean(), 3))
print("Avg F1:", round(df_results["f1"].mean(), 3), "| Median F1:", round(df_results["f1"].median(), 3))
print("Avg ROUGE-L F:", round(df_results["rougeL_f"].mean(), 3))
print("Exact Match rate:", round(df_results["exact_match"].mean(), 3))
print("Avg Numeric Recall:", round(df_results["numeric_recall"].mean(), 3))
print("Avg Unit Recall:", round(df_results["unit_recall"].mean(), 3))

print("\n=== WORST EXAMPLES (by F1) ===")
worst = df_results.sort_values("f1").head(10)
for _, r in worst.iterrows():
    print("-" * 80)
    print("Q:", r["question"])
    print("GT:", r["ground_truth"])
    print("PRED:", r["prediction"])
    print(f"F1={r['f1']:.3f} | ROUGE-L={r['rougeL_f']:.3f} | EM={r['exact_match']:.0f} | "
          f"NumRec={r['numeric_recall']:.3f} | UnitRec={r['unit_recall']:.3f} | Pages={r['retrieved_pages']}")

=== FINAL EVALUATION RESULTS (Revised) ===
Count: 3
Avg Precision: 0.009
Avg Recall: 0.967
Avg F1: 0.019 | Median F1: 0.016
Avg ROUGE-L F: 0.019
Exact Match rate: 0.0
Avg Numeric Recall: 0.667
Avg Unit Recall: 1.0

=== WORST EXAMPLES (by F1) ===
--------------------------------------------------------------------------------
Q: Who enforces the National Building Code of the Philippines?
GT: Not specified in the document
PRED: You are answering using ONLY the context. Rules: - Output ONE line only. - If the exact answer is not in context, output exactly: Not specified in the document. - Do not add notes, explanations, or extra commentary. - Do not paraphrase numbers; keep exact numbers and units. - If answering with a value, copy the exact clause/sentence from context that contains the value. Context: [Page 1] The National Building Code (P.D. 1096) 1 NATIONAL BUILDING CODE OF THE PHILIPPINES MALACAÑANG Manila PRESIDENTIAL DECREE NO. 1096 ADOPTING A NATIONAL BUILDING CODE OF THE PHILIPPI